In [42]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install --upgrade pip -quiet
!pip install --upgrade google-cloud-vision -quiet
!pip install --upgrade opencv-python --break-system-packages -quiet
!pip install pandas==2.2.2

print("Completed upgrading required packages...")
os.kill(os.getpid(),9)


Usage:   
  pip install [options] <requirement specifier> [package-index-options] ...
  pip install [options] -r <requirements file> [package-index-options] ...
  pip install [options] [-e] <vcs project url> ...
  pip install [options] [-e] <local project path> ...
  pip install [options] <archive url/path> ...

no such option: -u

Usage:   
  pip install [options] <requirement specifier> [package-index-options] ...
  pip install [options] -r <requirements file> [package-index-options] ...
  pip install [options] [-e] <vcs project url> ...
  pip install [options] [-e] <local project path> ...
  pip install [options] <archive url/path> ...

no such option: -u

Usage:   
  pip install [options] <requirement specifier> [package-index-options] ...
  pip install [options] -r <requirements file> [package-index-options] ...
  pip install [options] [-e] <vcs project url> ...
  pip install [options] [-e] <local project path> ...
  pip install [options] <archive url/path> ...

no such option: -

In [2]:
from google.cloud import vision
import io
import os

import numpy as np
import pandas as pd

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/kaggle/input/musait/manifest-canto-466706-v0-1bb68556dfe3.json"

# MVSA-Single

In [43]:
root = "/kaggle/input/mvsasingle/MVSA_Single/data"
labels = pd.read_csv("/kaggle/input/mvsasingle/MVSA_Single/labelResultAll.txt",sep=r'[,\t \n]+',index_col=False, engine="python")

In [44]:
# print(label.iloc[0])
sentiments = [] # image sentiments only
L = len(labels)

for i in range(0,L):
    # print(labels.iloc[i]["ID"])
    # print(type(labels.iloc[i]))
    sentiments+=[labels.iloc[i]["image"]]
    # break
print("Total number of files in MVSA-Single =",len(sentiments))
# print(sentiments)

Total number of files in MVSA-Single = 4869


In [45]:
from PIL import Image
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as patches

client = vision.ImageAnnotatorClient()

In [46]:
bool([0])

True

In [47]:
import cv2
from tqdm.notebook import tqdm

all_files = os.listdir(root)
images = [file for file in all_files if "jpg" in file]
images.sort()

text_detected, filename, filepath, ocr, coordinates = [],[],[],[],[]
txt_height, txt_width, img_height, img_width = [],[],[],[]


for image_name in tqdm(images, desc="Processing images (MVSA-Single)", unit="image"):
    # if image_name!="100.jpg":
    #     continue
    # print(image_name)
    # break
    fpath = os.path.join(root, image_name)

    with io.open(fpath, 'rb') as image_file:
        content = image_file.read()

    img_w, img_h = (Image.open(io.BytesIO(content))).size
    # print(img_w, img_h)
    img_width+=[img_w]
    img_height+=[img_h]

    coords,h,w = [],0,0
    
    image = vision.Image(content=content)
    response = client.text_detection(image=image)
    texts = response.text_annotations

    if texts:
        text = texts[0].description
        bounding_poly = texts[0].bounding_poly
        for vertex in bounding_poly.vertices:
            coords.append({"x": vertex.x, "y": vertex.y})
        x_vals, y_vals = [],[]
        x_vals = [p['x'] for p in coords]
        y_vals = [p['y'] for p in coords]
        x_min, y_min, x_max, y_max = min(x_vals),min(y_vals),max(x_vals),max(y_vals)
        h,w = y_max - y_min,x_max - x_min
    else:
        text = ""
    txt_height+=[h]
    txt_width+=[w]
    
    filename+=[image_name]
    filepath+=[fpath]
    flag = bool(texts)
    text_detected+=[flag]
    
    ocr+=[text]
    coordinates+=[coords]


print("DONE......................!!!")


    # print("Google Vision OCR")
    # print("______________________________")
    # print(texts[0])

    # print("\nOur derivatives")
    # print("______________________________")
    # print(f"Filename: {image_name}")
    # print(f"Filepath: {fpath}")
    # print(f"Text detected: {flag}")
    # print(f"OCR: {text}")
    # print(f"Coordinates: {coords}")
    # print(f"Txt H: {h}")
    # print(f"Txt W: {w}")
    # print(f"Img H: {img_h}")
    # print(f"Img W: {img_w}")
    
    # print("______________________________\n")
    # img = Image.open(fpath)
    # plt.imshow(img)
    # plt.axis('off')
    # plt.savefig("mvsa-single-example.png")
    # plt.show()
    
    
    # img = cv2.imread(fpath)
    # Draw rectangle on image
    # cv2.rectangle(img, (x_min, y_min), (x_max, y_max), color=(255, 0, 0), thickness=2)
    
    # Show image
    # plt.imshow(img)
    # plt.axis('off')
    # plt.title("Bounding Box on Image")
    # plt.savefig("mvsa-single-example-bdBox.png")
    # plt.show()
    # break

Processing images (MVSA-Single):   0%|          | 0/4869 [00:00<?, ?image/s]

DONE......................!!!


In [48]:
# text_detected, filename, filepath, ocr, coordinates = [],[],[],[],[]
# txt_height, txt_width, img_height, img_width = [],[],[],[]
mvsa_single_info = pd.DataFrame({
    "filename":filename,
    "filepath":filepath,
    "img_height":img_height,
    "img_width":img_width,
    "text_detected":text_detected,
    "ocr":ocr,
    "coordinates":coordinates,
    "text_height":txt_height,
    "text_width":txt_width,
    "sentiment":sentiments
})

mvsa_single_info.to_csv("mvsa_single_info.csv", index=False)

In [49]:
len(mvsa_single_info[mvsa_single_info["text_detected"]==True])

2813

In [50]:
# Define standard aspect ratios
standard_ratios = {
    "1:1": 1.0, # 1 BOX
    "4:3": 4/3, # 1.33 Horizontal
    "3:2": 3/2, # 1.5 Horizontal
    "16:9": 16/9, # 1.78 Horizontal
    "17:9": 17/9, # 1.93 Horizontal
    "5:4": 5/4, # 1.25 Horizontal
    "21:9": 21/9, # 2.33 Horizontal
    "2:1": 2.0, # 2 Horizontal
    "7:5": 7/5, # 1.4 Horizontal
    # "A4 (1.41:1)": 1.41,
    "3:4": 3/4, # 0.75 Vertical
    "2:3": 2/3, # 0.67 Vertical
    "9:16": 9/16, # 0.5625 Vertical
    "5:7": 5/7, # 0.71 Vertical
    "4:5": 4/5, # 0.8 Vertical
    "A-series (1:√2)": 1/1.414, # 0.707 Vertical
    "2:5": 2/5, # 0.4 Vertical
    "1:3": 1/3, # 0.33 Vertical
    "Other":0
}

# Function to match image to closest standard aspect ratio
def closest_standard_ratio(width, height, tolerance=0.10):
    if height == 0:
        return "Invalid"
    ratio = width / height
    for name, std_ratio in standard_ratios.items():
        if abs(ratio - std_ratio) <= tolerance:
            return name
    return "Other"


ar_columnNames = list(standard_ratios.keys())
rowNames = ["positive", "negative", "neutral"]

txt_ar_df = pd.DataFrame(0, index=rowNames, columns=ar_columnNames)
img_ar_df = pd.DataFrame(0, index=rowNames, columns=ar_columnNames)

aspect_ratio_orientation = {
    # --- Box-like (almost square) ---
    "1:1": "box",

    # --- Horizontal (landscape) ---
    "4:3": "horizontal",      # 1.33
    "3:2": "horizontal",      # 1.5
    "16:9": "horizontal",     # 1.78
    "17:9": "horizontal",     # 1.93
    "5:4": "horizontal",      # 1.25
    "21:9": "horizontal",     # 2.33
    "2:1": "horizontal",      # 2.0
    "7:5": "horizontal",      # 1.4
    "A4 (1.41:1)": "horizontal",  # 1.41

    # --- Vertical (portrait) ---
    "3:4": "vertical",        # 0.75
    "2:3": "vertical",        # 0.666
    "9:16": "vertical",       # 0.5625
    "5:7": "vertical",        # 0.714
    "4:5": "vertical",        # 0.8
    "A-series (1:√2)": "vertical",  # ≈0.707
    "2:5": "vertical",        # 0.4
    "1:3": "vertical",

    # --- Catch-all ---
    # "Other": "other"
    "Other": "box"
}



# Helper function to classify orientation
def get_orientation(aspect_ratio="Other"):
    return aspect_ratio_orientation[aspect_ratio]




columnNames = ["horizontal", "box","vertical"]
rowNames = ["positive", "negative", "neutral"]

# Create a DataFrame filled with zeros
txt_orientation_df = pd.DataFrame(0, index=rowNames, columns=columnNames)
img_orientation_df = pd.DataFrame(0, index=rowNames, columns=columnNames)

In [51]:
img_w_txt=0

L = len(mvsa_single_info)

for i in tqdm(range(L), desc="Orientation Analysis (MVSA)", unit="image"):
    if not mvsa_single_info.iloc[i]["text_detected"]: # False
        continue
    img_w_txt+=1
    row = mvsa_single_info.iloc[i]
    img_w, img_h = row["img_width"],row["img_height"]
    txt_w, txt_h = row["text_width"],row["text_height"]
    
    txt_ar = closest_standard_ratio(txt_w, txt_h)
    img_ar = closest_standard_ratio(img_w, img_h)
    
    txt_orientation = aspect_ratio_orientation[txt_ar]
    img_orientation = aspect_ratio_orientation[img_ar]
    
    img_ar_df.loc[row['sentiment'],img_ar]+=1
    txt_ar_df.loc[row['sentiment'],txt_ar]+=1
    
    img_orientation_df.loc[row['sentiment'],img_orientation]+=1
    txt_orientation_df.loc[row['sentiment'],txt_orientation]+=1

print("Done!")

Orientation Analysis (MVSA):   0%|          | 0/4869 [00:00<?, ?image/s]

Done!


In [52]:
img_w_txt

2813

In [53]:
img_orientation_df

,horizontal,box,vertical
positive,644,466,460
negative,290,191,213
neutral,222,155,172


In [54]:
from scipy.stats import chi2_contingency

def chisq_test_orientation(data, alpha=0.05):
    sentiments = ['positive', 'negative', 'neutral']
    orientations = ['horizontal', 'box', 'vertical']
    
    chi2, p, dof, expected = chi2_contingency(data)
    print()
    print(f"Chi-square statistic = {chi2:.3f}")
    print(f"p-value = {p}")
    print(f"Degrees of freedom = {dof}")
    print()
    print("Expected frequencies:")
    print(pd.DataFrame(np.round(expected, 2), index=sentiments, columns=orientations))
    # Decision
    # alpha = 0.05
    if p < alpha:
        print("Reject the null hypothesis: There is a relationship between orientation and sentiment.")
    else:
        print("Fail to reject the null hypothesis: No significant relationship found.")

In [55]:
datasets = [img_orientation_df, txt_orientation_df]

for i,data in enumerate(datasets):
    print("_"*50)
    print(f"Dataset {i}")
    print("_"*100)
    print("Original frequencies:\n",data)
    los = 0.05
    print(f"Chosen level of significance = {los}")
    chisq_test_orientation(data, los)

__________________________________________________
Dataset 0
____________________________________________________________________________________________________
Original frequencies:
           horizontal  box  vertical
positive         644  466       460
negative         290  191       213
neutral          222  155       172
Chosen level of significance = 0.05

Chi-square statistic = 1.702
p-value = 0.7903451704609561
Degrees of freedom = 4

Expected frequencies:
          horizontal     box  vertical
positive      645.19  453.20    471.61
negative      285.20  200.33    208.47
neutral       225.61  158.47    164.91
Fail to reject the null hypothesis: No significant relationship found.
__________________________________________________
Dataset 1
____________________________________________________________________________________________________
Original frequencies:
           horizontal  box  vertical
positive         464  761       345
negative         215  310       169
neutral   

# MVSA-Multiple

In [3]:
import re
from collections import Counter

label_file = "/kaggle/input/mvsamultiple/MVSA/labelResultAll.txt"  # adjust path as needed

def get_majority_text(text_list):
    counts = Counter(text_list).most_common()
    if len(counts) > 1 and counts[0][1] == counts[1][1]:
        return "neutral"  # Tie case
    return counts[0][0]

sentiments = []
count = 0

with open(label_file, "r", encoding="utf-8") as f:
    for line in f:
        if "ID" in line:
            continue  # skip header
        count+=1
        parts = re.split(r'[\n\t]', line.strip())
        parts = list(filter(None, parts))  # remove empty strings
        annotators = parts[1:]  # ignore the ID
        img_sent = [ann.split(",")[1] for ann in annotators]  # get image sentiment
        majority_sentiment = get_majority_text(img_sent)
        sentiments.append(majority_sentiment)

# Example: print first few results
print(sentiments[:5])

['positive', 'positive', 'neutral', 'positive', 'neutral']


In [4]:
len(sentiments)

19600

In [5]:
from PIL import Image
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as patches

client = vision.ImageAnnotatorClient()

In [6]:
# bool([0])
root = "/kaggle/input/mvsamultiple/MVSA/data"

In [8]:
import cv2
from tqdm.notebook import tqdm

all_files = os.listdir(root)
images = [file for file in all_files if "jpg" in file]
images.sort()

text_detected, filename, filepath, ocr, coordinates = [],[],[],[],[]
txt_height, txt_width, img_height, img_width = [],[],[],[]


for image_name in tqdm(images, desc="Processing images (MVSA-Multiple)", unit="image"):
    # if image_name!="100.jpg":
    #     continue
    # print(image_name)
    # break
    fpath = os.path.join(root, image_name)

    with io.open(fpath, 'rb') as image_file:
        content = image_file.read()
    img_w, img_h = None,None
    try:
        img_w, img_h = (Image.open(io.BytesIO(content))).size
        continue
        # img_w, img_h = img.size
    except:
        indices+=[index]
        continue
    # img_w, img_h = (Image.open(io.BytesIO(content))).size
    # print(img_w, img_h)
    img_width+=[img_w]
    img_height+=[img_h]

    coords,h,w = [],0,0
    
    image = vision.Image(content=content)
    response = client.text_detection(image=image)
    texts = response.text_annotations

    if texts:
        text = texts[0].description
        bounding_poly = texts[0].bounding_poly
        for vertex in bounding_poly.vertices:
            coords.append({"x": vertex.x, "y": vertex.y})
        x_vals, y_vals = [],[]
        x_vals = [p['x'] for p in coords]
        y_vals = [p['y'] for p in coords]
        x_min, y_min, x_max, y_max = min(x_vals),min(y_vals),max(x_vals),max(y_vals)
        h,w = y_max - y_min,x_max - x_min
    else:
        text = ""
    txt_height+=[h]
    txt_width+=[w]
    
    filename+=[image_name]
    filepath+=[fpath]
    flag = bool(texts)
    text_detected+=[flag]
    
    ocr+=[text]
    coordinates+=[coords]


print("DONE......................!!!")


    # print("Google Vision OCR")
    # print("______________________________")
    # print(texts[0])

    # print("\nOur derivatives")
    # print("______________________________")
    # print(f"Filename: {image_name}")
    # print(f"Filepath: {fpath}")
    # print(f"Text detected: {flag}")
    # print(f"OCR: {text}")
    # print(f"Coordinates: {coords}")
    # print(f"Txt H: {h}")
    # print(f"Txt W: {w}")
    # print(f"Img H: {img_h}")
    # print(f"Img W: {img_w}")
    
    # print("______________________________\n")
    # img = Image.open(fpath)
    # plt.imshow(img)
    # plt.axis('off')
    # plt.savefig("mvsa-single-example.png")
    # plt.show()
    
    
    # img = cv2.imread(fpath)
    # Draw rectangle on image
    # cv2.rectangle(img, (x_min, y_min), (x_max, y_max), color=(255, 0, 0), thickness=2)
    
    # Show image
    # plt.imshow(img)
    # plt.axis('off')
    # plt.title("Bounding Box on Image")
    # plt.savefig("mvsa-single-example-bdBox.png")
    # plt.show()
    # break

Processing images (MVSA-Multiple):   0%|          | 0/19600 [00:00<?, ?image/s]

DONE......................!!!


In [26]:
filename[:5]

['10000.jpg', '10001.jpg', '10002.jpg', '10003.jpg', '10004.jpg']

In [30]:
missing = []

for image_name in tqdm(images, desc=f"Finding corrupted images (MVSA-Multiple):", unit="image"):
    if image_name not in filename:
        missing+=[image_name]

print(f"Missing filenames are: {missing}")

Finding corrupted images (MVSA-Multiple)::   0%|          | 0/19600 [00:00<?, ?image/s]

Missing filenames are: ['3151.jpg', '3910.jpg', '5995.jpg']


In [33]:
# images.index('3151.jpg')

for name in missing:
    index = images.index(name)
    del sentiments[index]

print(f"Length of sentiments list: {len(sentiments)}")

Length of sentiments list: 19597


In [34]:
# text_detected, filename, filepath, ocr, coordinates = [],[],[],[],[]
# txt_height, txt_width, img_height, img_width = [],[],[],[]
mvsa_multiple_info = pd.DataFrame({
    "filename":filename,
    "filepath":filepath,
    "img_height":img_height,
    "img_width":img_width,
    "text_detected":text_detected,
    "ocr":ocr,
    "coordinates":coordinates,
    "text_height":txt_height,
    "text_width":txt_width,
    "sentiment":sentiments
})

mvsa_multiple_info.to_csv("mvsa_multiple_info.csv", index=False)

In [35]:
len(mvsa_multiple_info[mvsa_multiple_info["text_detected"]==True])

12866

In [36]:
# Define standard aspect ratios
standard_ratios = {
    "1:1": 1.0, # 1 BOX
    "4:3": 4/3, # 1.33 Horizontal
    "3:2": 3/2, # 1.5 Horizontal
    "16:9": 16/9, # 1.78 Horizontal
    "17:9": 17/9, # 1.93 Horizontal
    "5:4": 5/4, # 1.25 Horizontal
    "21:9": 21/9, # 2.33 Horizontal
    "2:1": 2.0, # 2 Horizontal
    "7:5": 7/5, # 1.4 Horizontal
    # "A4 (1.41:1)": 1.41,
    "3:4": 3/4, # 0.75 Vertical
    "2:3": 2/3, # 0.67 Vertical
    "9:16": 9/16, # 0.5625 Vertical
    "5:7": 5/7, # 0.71 Vertical
    "4:5": 4/5, # 0.8 Vertical
    "A-series (1:√2)": 1/1.414, # 0.707 Vertical
    "2:5": 2/5, # 0.4 Vertical
    "1:3": 1/3, # 0.33 Vertical
    "Other":0
}

# Function to match image to closest standard aspect ratio
def closest_standard_ratio(width, height, tolerance=0.10):
    if height == 0:
        return "Invalid"
    ratio = width / height
    for name, std_ratio in standard_ratios.items():
        if abs(ratio - std_ratio) <= tolerance:
            return name
    return "Other"


ar_columnNames = list(standard_ratios.keys())
rowNames = ["positive", "negative", "neutral"]

txt_ar_df = pd.DataFrame(0, index=rowNames, columns=ar_columnNames)
img_ar_df = pd.DataFrame(0, index=rowNames, columns=ar_columnNames)

aspect_ratio_orientation = {
    # --- Box-like (almost square) ---
    "1:1": "box",

    # --- Horizontal (landscape) ---
    "4:3": "horizontal",      # 1.33
    "3:2": "horizontal",      # 1.5
    "16:9": "horizontal",     # 1.78
    "17:9": "horizontal",     # 1.93
    "5:4": "horizontal",      # 1.25
    "21:9": "horizontal",     # 2.33
    "2:1": "horizontal",      # 2.0
    "7:5": "horizontal",      # 1.4
    "A4 (1.41:1)": "horizontal",  # 1.41

    # --- Vertical (portrait) ---
    "3:4": "vertical",        # 0.75
    "2:3": "vertical",        # 0.666
    "9:16": "vertical",       # 0.5625
    "5:7": "vertical",        # 0.714
    "4:5": "vertical",        # 0.8
    "A-series (1:√2)": "vertical",  # ≈0.707
    "2:5": "vertical",        # 0.4
    "1:3": "vertical",

    # --- Catch-all ---
    # "Other": "other"
    "Other": "box"
}



# Helper function to classify orientation
def get_orientation(aspect_ratio="Other"):
    return aspect_ratio_orientation[aspect_ratio]




columnNames = ["horizontal", "box","vertical"]
rowNames = ["positive", "negative", "neutral"]

# Create a DataFrame filled with zeros
txt_orientation_df = pd.DataFrame(0, index=rowNames, columns=columnNames)
img_orientation_df = pd.DataFrame(0, index=rowNames, columns=columnNames)

In [38]:
img_w_txt=0

L = len(mvsa_multiple_info)

for i in tqdm(range(L), desc="Orientation Analysis (MVSA-Multiple)", unit="image"):
    if not mvsa_multiple_info.iloc[i]["text_detected"]: # False
        continue
    img_w_txt+=1
    row = mvsa_multiple_info.iloc[i]
    img_w, img_h = row["img_width"],row["img_height"]
    txt_w, txt_h = row["text_width"],row["text_height"]
    
    txt_ar = closest_standard_ratio(txt_w, txt_h)
    img_ar = closest_standard_ratio(img_w, img_h)
    
    txt_orientation = aspect_ratio_orientation[txt_ar]
    img_orientation = aspect_ratio_orientation[img_ar]
    
    img_ar_df.loc[row['sentiment'],img_ar]+=1
    txt_ar_df.loc[row['sentiment'],txt_ar]+=1
    
    img_orientation_df.loc[row['sentiment'],img_orientation]+=1
    txt_orientation_df.loc[row['sentiment'],txt_orientation]+=1

print("Done!")

Orientation Analysis (MVSA-Multiple):   0%|          | 0/19597 [00:00<?, ?image/s]

Done!


In [14]:
img_w_txt

2813

In [39]:
img_orientation_df

,horizontal,box,vertical
positive,2715,1879,1996
negative,343,258,291
neutral,2263,1536,1585


In [40]:
from scipy.stats import chi2_contingency

def chisq_test_orientation(data, alpha=0.05):
    sentiments = ['positive', 'negative', 'neutral']
    orientations = ['horizontal', 'box', 'vertical']
    
    chi2, p, dof, expected = chi2_contingency(data)
    print()
    print(f"Chi-square statistic = {chi2:.3f}")
    print(f"p-value = {p}")
    print(f"Degrees of freedom = {dof}")
    print()
    print("Expected frequencies:")
    print(pd.DataFrame(np.round(expected, 2), index=sentiments, columns=orientations))
    # Decision
    # alpha = 0.05
    if p < alpha:
        print("Reject the null hypothesis: There is a relationship between orientation and sentiment.")
    else:
        print("Fail to reject the null hypothesis: No significant relationship found.")

In [41]:
datasets = [img_orientation_df, txt_orientation_df]

for i,data in enumerate(datasets):
    print("_"*50)
    print(f"Dataset {i}")
    print("_"*100)
    print("Original frequencies:\n",data)
    los = 0.05
    print(f"Chosen level of significance = {los}")
    chisq_test_orientation(data, los)

__________________________________________________
Dataset 0
____________________________________________________________________________________________________
Original frequencies:
           horizontal   box  vertical
positive        2715  1879      1996
negative         343   258       291
neutral         2263  1536      1585
Chosen level of significance = 0.05

Chi-square statistic = 5.246
p-value = 0.26299390634082254
Degrees of freedom = 4

Expected frequencies:
          horizontal      box  vertical
positive     2725.43  1881.32   1983.25
negative      368.91   254.65    268.45
neutral      2226.66  1537.03   1620.31
Fail to reject the null hypothesis: No significant relationship found.
__________________________________________________
Dataset 1
____________________________________________________________________________________________________
Original frequencies:
           horizontal   box  vertical
positive        1941  3085      1564
negative         263   410       21